# Flowers Knowledge RAG Pipeline
Step-by-step RAG demo using LangChain + Google Gemini + ChromaDB

## Step 1 - Install Dependencies

In [ ]:
%pip install langchain langchain-community langchain-google-genai langchain-chroma langchain-text-splitters google-generativeai python-dotenv

## Step 2 - Imports

In [ ]:
import os
from dotenv import load_dotenv

# document loaders is used to load lot of documents in single run
# TextLoader because we are loading text files, for pdf use PDFLoader
from langchain_community.document_loaders import TextLoader

# to create chunks we use langchain_text_splitter
# Splitting text by recursively looking at characters
from langchain_text_splitters import RecursiveCharacterTextSplitter

# we use chroma db to store the vectors, it will create vectors in our local laptop
from langchain_chroma import Chroma

# this we use to generate the embeddings and chat with Gemini LLM
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

print("All libraries imported successfully")

## Step 3 - Load API Key

In [ ]:
load_dotenv(override=True)
api_key = os.environ.get("GOOGLE_API_KEY")
print("API Key loaded:", "OK" if api_key else "NOT FOUND - check your .env file")

## Step 4 - Load flowers.txt (Knowledge Base)

In [ ]:
loader = TextLoader("flowers.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total characters: {len(documents[0].page_content)}")

## Step 5 - Split Documents into Chunks

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
docs = splitter.split_documents(documents)

print(f"Total chunks created: {len(docs)}")
print(f"\nSample chunk:\n{docs[0].page_content[:300]}")

## Step 6 - Create Vector Database with Gemini Embeddings

In [ ]:
vectorstore = Chroma.from_documents(
    docs,
    GoogleGenerativeAIEmbeddings(
        model="models/gemini-embedding-001",
        google_api_key=api_key
    )
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

print("Vector database created successfully")

## Step 7 - Initialize Gemini LLM

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-flash-latest",
    temperature=0.3,
    google_api_key=api_key
)

print("Gemini LLM initialized")

## Step 8 - Build the RAG Function

In [ ]:
def combine_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def ask_rag(question):
    # Step 1: Retrieve relevant chunks from vectorstore
    retrieved_docs = retriever.invoke(question)
    context = combine_docs(retrieved_docs)

    # Step 2: Build prompt with context + question
    prompt = f"""
You are a factual question-answering assistant.
Use ONLY the information provided in the context.
If the answer is not present, say: I don't know.

Context:
{context}

Question: {question}

Answer:
"""
    # Step 3: Get answer from Gemini LLM
    response = llm.invoke(prompt)
    content = response.content
    if isinstance(content, list):
        return "".join(block.get("text", "") for block in content if isinstance(block, dict))
    return content

print("RAG function ready")

## Step 9 - Ask Questions

In [ ]:
question = "What is the purpose of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
question = "What are the parts of a flower?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")

In [ ]:
question = "What is pollination?"
answer = ask_rag(question)
print(f"Q: {question}")
print(f"A: {answer}")